In [ ]:
# Re-import edited fund_clustering modules automatically before each cell runs.
%load_ext autoreload
%autoreload 2

In [ ]:
# Load the newest weekly export: allocations and metadata, equity funds only.
import sys
from pathlib import Path

from fund_clustering.load import latest_week, load_week

# Should point into FundClustering\.venv; anything else means the wrong kernel.
print(sys.executable)

DATA_DIR = Path("../data")  # git-ignored copies of the CSV exports
WEEK = latest_week(DATA_DIR)

data = load_week(DATA_DIR, WEEK)

# Week, (funds, columns), country column count, sector column count
WEEK, data.funds.shape, len(data.country_columns), len(data.sector_columns)


In [ ]:
# Cosine features: one vector per fund (countries and sectors), scaled to length 1
# so funds are compared by allocation mix, not size.
import numpy as np

from fund_clustering.methods.cosine import cosine_features

X = cosine_features(data)

# Sanity check: every row should have length 1 (tiny rounding is fine).
lengths = np.linalg.norm(X, axis=1)

X.shape, lengths.min(), lengths.max()


In [ ]:
# Clustering: 10D UMAP + HDBSCAN gives a cluster id per fund (-1 = not in a
# cluster); a separate 2D UMAP gives the map position. ~20 s on the first run.
from fund_clustering.cluster import cluster_funds

result = cluster_funds(X)

# Working table for the cells below: fund data plus cluster and map position.
funds = data.funds.assign(
    cluster=result.labels, x=result.xy[:, 0], y=result.xy[:, 1]
)

# (number of clusters, funds not in a cluster)
result.cluster_count, result.not_in_cluster_count


In [ ]:
# Naming: each cluster is named after its most common category ("Mixed · A / B"
# under 50% purity); each fund gets a verdict comparing its category to that.
from fund_clustering.naming import describe_clusters, fund_verdicts

clusters = describe_clusters(funds["category"], result.labels)
funds["verdict"] = fund_verdicts(funds["category"], result.labels, clusters)
funds["cluster_name"] = funds["cluster"].map(clusters["name"]).fillna("Not in a cluster")

# agree / neighbouring (same regional family) / different (the other family) / none
display(funds["verdict"].value_counts())

# Largest clusters first; purity = share of the leading category.
clusters[["name", "size", "purity"]].sort_values("size", ascending=False)


In [ ]:
# Neighbours: the 20 most similar funds per fund, measured on the real allocation
# vectors (not the 2D map). Isolated = 5% of funds farthest from their 5th neighbour.
import pandas as pd

from fund_clustering.neighbours import isolated_flags, nearest_neighbours

neighbours = nearest_neighbours(X)
funds["isolated"] = isolated_flags(neighbours)


def similar_funds(isin: str, count: int = 10) -> pd.DataFrame:
    """Most similar funds to ``isin``, closest first (distance 0 = identical mix)."""
    row = funds.index[funds["isin"] == isin][0]
    similar = funds.iloc[neighbours.indices[row, :count]]
    return similar[["isin", "name", "category", "cluster_name"]].assign(
        distance=neighbours.distances[row, :count].round(3)
    )


# Try any ISIN here. This one is the energy-fund test case from the note.
ISIN = "LU0256331488"
selected = funds.loc[funds["isin"] == ISIN].iloc[0]
print(selected["name"], "|", selected["category"], "|", selected["cluster_name"])
similar_funds(ISIN)


In [ ]:
# Territory: map background per cluster. A ~220-cell grid; each cell takes the
# distance-weighted vote of its 7 nearest clustered funds within a radius,
# then each cluster's cells are merged into polygons (drawn in the map below).
import shapely

from fund_clustering.territory import territory_grid, territory_polygons

grid = territory_grid(result.xy, result.labels)
territories = territory_polygons(grid)

# (grid rows x cols, territories, polygon parts, total vertices to export)
parts = sum(len(getattr(shape, "geoms", [shape])) for shape in territories.values())
vertices = sum(shapely.get_num_coordinates(shape) for shape in territories.values())
grid.labels.shape, len(territories), parts, vertices


In [ ]:
# Map drawing used by the next cell: one dot per fund, coloured by cluster, over
# its cluster's territory. Scroll to zoom, drag to pan, hover for details. Only
# local closeness is meaningful, not distances between far-apart groups.
import plotly.express as px
import plotly.graph_objects as go

# Outline colour by verdict: amber = neighbouring, red = different family.
VERDICT_OUTLINE = {
    "agree": "rgba(0,0,0,0)",
    "neighbouring": "#c58300",
    "different": "#d0353a",
    "none": "rgba(0,0,0,0)",
}

# One colour per cluster, shared by dots and territory; grey = not in a cluster.
PALETTE = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24


def with_alpha(hex_colour: str, alpha: float) -> str:
    r, g, b = (int(hex_colour[i : i + 2], 16) for i in (1, 3, 5))
    return f"rgba({r},{g},{b},{alpha})"


def map_figure(
    table: pd.DataFrame,
    clusters: pd.DataFrame,
    territories: dict,
    title: str,
    subtitle: str = "",
) -> go.Figure:
    """Cluster map for one method; ``table`` needs cluster, x, y and verdict."""
    cluster_colours = {
        str(cluster_id): PALETTE[i % len(PALETTE)]
        for i, cluster_id in enumerate(sorted(clusters.index))
    }
    cluster_colours["-1"] = "#9aa0a6"

    fig = px.scatter(
        table.assign(cluster=table["cluster"].astype(str)),  # str -> discrete colours
        x="x",
        y="y",
        color="cluster",
        color_discrete_map=cluster_colours,
        hover_name="name",
        hover_data=[
            "isin", "category", "cluster_name", "verdict", "isolated", "company_name"
        ],
        height=750,
    )
    # plotly makes one trace per cluster; give each trace's dots their outlines.
    for trace in fig.data:
        rows = table["cluster"].astype(str) == trace.name
        trace.marker.line = dict(
            color=table.loc[rows, "verdict"].map(VERDICT_OUTLINE).tolist(), width=2
        )

    # Territory polygons (outer edges only), added last, then moved behind the dots.
    dot_traces = len(fig.data)
    for cluster_id, shape in territories.items():
        colour = cluster_colours[str(cluster_id)]
        for polygon in getattr(shape, "geoms", [shape]):
            x, y = polygon.exterior.xy
            fig.add_trace(
                go.Scatter(
                    x=list(x),
                    y=list(y),
                    fill="toself",
                    fillcolor=with_alpha(colour, 0.22),
                    line=dict(color=with_alpha(colour, 0.6), width=0.5),
                    mode="lines",
                    hoverinfo="skip",
                    showlegend=False,
                )
            )
    fig.data = fig.data[dot_traces:] + fig.data[:dot_traces]

    fig.update_layout(title=dict(text=title, subtitle=dict(text=subtitle)))
    fig.update_yaxes(scaleanchor="x")  # same scale on both axes, no stretching
    return fig


In [ ]:
# Every registered method end to end, one map each, captioned with the method's
# name, cluster counts and description. ~20 s per method on the first run.
from fund_clustering.methods import METHODS
from fund_clustering.pipeline import run_method

runs = {method_id: run_method(data, method) for method_id, method in METHODS.items()}

for run in runs.values():
    table = data.funds.assign(
        cluster=run.result.labels,
        x=run.result.xy[:, 0],
        y=run.result.xy[:, 1],
        verdict=run.verdicts,
        isolated=run.isolated,
    )
    table["cluster_name"] = (
        table["cluster"].map(run.clusters["name"]).fillna("Not in a cluster")
    )
    title = (
        f"{run.method.name}: {run.result.cluster_count} clusters, "
        f"{run.result.not_in_cluster_count} not in a cluster"
    )
    map_figure(
        table, run.clusters, run.territories, title, run.method.description
    ).show()
